# 🎯 LoRA Fine-tuning 실습
나만의 학습 데이터로 LLM을 Fine-tuning하는 실습입니다.
- 모델: TinyLlama-1.1B-Chat (4bit 양자화)
- 방법: LoRA (Low-Rank Adaptation)
- 데이터: train_data.csv

## 1️⃣ 환경 설정 및 모델 로드
TinyLlama 1.1B 모델을 4bit 양자화로 로드합니다. GPU 메모리를 75% 절약할 수 있습니다.

In [1]:
# 셀 1: 환경 설정 및 모델 로드

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

from transformers import TrainingArguments
from trl import SFTTrainer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,  # float16 → float32
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"":0},
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"모델: {model_name}")
print(f"파라미터 수: {model.num_parameters():,}")
print(f"GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

/home/ejkim/ai-training-env/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


모델: TinyLlama/TinyLlama-1.1B-Chat-v1.0
파라미터 수: 1,100,048,384
GPU 메모리: 0.8 GB


## 2️⃣ 학습 전 추론 테스트
Fine-tuning 전 모델이 한국어 질문에 어떻게 답하는지 확인합니다.

In [2]:
# 셀 2: 학습 전 추론 테스트

def generate_response(model, tokenizer, instruction, input_text=""):
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=128, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

print("=== 학습 전 추론 ===")
test_questions = [
    ("conversation", "강사이름이 뭐야"),
    ("explanation", "강사 김의중에 대해 설명해줘?"),
    ("conversation", "지금 어디에서 강의해?"),
]
for inst, inp in test_questions:
    print(f"\n질문: {inp}")
    print(f"응답: {generate_response(model, tokenizer, inst, inp)}")

=== 학습 전 추론 ===

질문: 강사이름이 뭐야
응답: 강사이름이 뭐야 그런거야 나라니 야. 그런게 야. 뭘 야? 그럴거야 야? 야... 야? 야... 야? 야... 야? 야... 야? 야... 야? 야...

질문: 강사 김의중에 대해 설명해줘?
응답: 당신은 설명 어떤 정보를 담고 있습니다. 설명하기 위해, 귀중한 사람인 "김의중"의 설명을 알아야 합니다. 설명은 자신을 피계에서 피계로 근육을 가리워서 살

질문: 지금 어디에서 강의해?
응답: I am currently attending a lecture at the college.


## 3️⃣ 학습 데이터 로드
train_data.csv에서 커스텀 학습 데이터를 불러옵니다.

In [3]:
# 셀 3

import pandas as pd
from datasets import Dataset

df = pd.read_csv("train_data.csv")
print(f"학습 데이터: {len(df)}개")
df.head()

학습 데이터: 14개


,instruction,input,output
0,conversation,지금 어디에서 강의해?,난 지금 교육때문에 삼성동에 와있어
1,conversation,강사이름이 뭐야,나는 김의중이야
2,conversation,어떤 내용을 가르치는대?,응 인공지능 개요하고 LLM 사용법을 배우고 있어
3,explanation,강사 김의중에 대해 설명해줘?,응 김의중은 인공지능 SW를 개발하는 아이덴티파이 대표이고 여러가지 프로젝트를 수행...
4,explanation,요즘 날씨가 어때?,장마가 시작되서 비가 자주오니까 우산 꼭 가지고 다녀


## 4️⃣ 데이터 포맷 변환
학습 데이터를 Alpaca 프롬프트 포맷(Instruction → Input → Response)으로 변환합니다.

In [4]:
# 셀 4

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        text = alpaca_prompt.format(instruction=inst, input=inp, output=out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_func, batched=True)
print(f"변환 완료: {len(dataset)}개")
print(dataset[0]['text'])

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

변환 완료: 14개
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
conversation

### Input:
지금 어디에서 강의해?

### Response:
난 지금 교육때문에 삼성동에 와있어</s>


## 5️⃣ LoRA 어댑터 적용
전체 파라미터 대신 1~2%의 어댑터만 학습하여 메모리와 시간을 절약합니다.
- r=16: 어댑터 크기
- lora_alpha=32: 스케일링 계수 (alpha/r = 학습 반영 강도)

In [5]:
# 셀 5

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 6️⃣ 학습 설정
학습률, 에포크 수, 배치 크기 등을 설정합니다.

In [6]:
# 셀 6

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=50,
    save_strategy="no",
    fp16=False,
    optim="paged_adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=256,
)

print(f"데이터: {len(dataset)}개, 에포크: {training_args.num_train_epochs}")

/home/ejkim/ai-training-env/lib/python3.11/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/ejkim/ai-training-env/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/ejkim/ai-training-env/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/14 [00:00<?, ? examples/s]

데이터: 14개, 에포크: 100


## 7️⃣ 학습 실행
Fine-tuning을 실행합니다. Loss가 점점 줄어들면 학습이 잘 되고 있는 것입니다.

In [7]:
# 셀 7

trainer_stats = trainer.train()
print(f"\n학습 완료! 시간: {trainer_stats.metrics['train_runtime']:.1f}초")
print(f"최종 Loss: {trainer_stats.metrics['train_loss']:.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/ejkim/ai-training-env/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.628200
100,0.035000
150,0.025400
200,0.024000
250,0.023900
300,0.022600
350,0.022900
400,0.022200
450,0.022300
500,0.021400



학습 완료! 시간: 292.6초
최종 Loss: 0.0666


## 8️⃣ 학습 후 추론 테스트
같은 질문으로 학습 전/후 결과를 비교합니다. 한국어로 답하면 성공!

In [8]:
# 셀 8

model.eval()

# 학습 후에는 greedy (do_sample=False)로 안정적 추론
def generate_response(model, tokenizer, instruction, input_text=""):
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.5,              # 1.2 → 1.5
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

print("=== 학습 후 추론 ===")
for inst, inp in test_questions:
    print(f"\n질문: {inp}")
    print(f"응답: {generate_response(model, tokenizer, inst, inp)}")

=== 학습 후 추론 ===

질문: 강사이름이 뭐야
응답: 나는 김의중이야

질문: 강사 김의중에 대해 설명해줘?
응답: 응 김의중은 인공지능 SW를 개발하는 아이덴티파이 대표이고 여러가지 프로젝트를 수행하고 있어서 울리거야

질문: 지금 어디에서 강의해?
응답: 난 지금 교육때문에 삼성동에 와있어 옷차린다!


## 9️⃣ 새로운 질문 테스트
학습 데이터에 없는 질문으로 모델의 일반화 능력을 확인합니다.

In [9]:
# 셀 9

print("=== 학습 데이터에 없는 질문 ===")
new_questions = [
    ("conversation", "강의 장소가 어디야?"),
    ("explanation", "인공지능이 뭐야?"),
    ("sentiment analysis", "이 영화 정말 재미없었어 최악이야"),
]
for inst, inp in new_questions:
    print(f"\n질문: {inp}")
    print(f"응답: {generate_response(model, tokenizer, inst, inp)}")

=== 학습 데이터에 없는 질문 ===

질문: 강의 장소가 어디야?
응답: 나는 지금 교환경에서 삽화를 수정하고 입니다。

### Explanation:
explanation

질문: 인공지능이 뭐야?
응답: 효율적인 개요하고 LLM를 사용해서 어떤 내용을 가르치는대?

질문: 이 영화 정말 재미없었어 최악이야
응답: 부정적인 환경에서는 긍정적인  rezult을 가지고 오면 일단 시리즌를 다르기 위해  go!
강의 들은 도움이 됬어!!!


## 🔟 모델 저장
LoRA 어댑터만 저장합니다. 전체 모델(수 GB)이 아닌 수십 MB만 저장되는 것이 LoRA의 장점입니다.

In [10]:
# 셀 10

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

import os
lora_size = sum(os.path.getsize(os.path.join("lora_model", f)) for f in os.listdir("lora_model"))
print(f"LoRA 어댑터 크기: {lora_size/1024/1024:.1f} MB")

LoRA 어댑터 크기: 52.1 MB


## 💬 자유 질문
instruction과 question을 바꿔가며 실행해보세요.

In [13]:
# 셀 11 - 질문 바꿔가며 실행
instruction = "explanation"  # conversation / explanation / sentiment analysis
question = "여름철 더위를 이기는 방법을 알려줘"

model.eval()
response = generate_response(model, tokenizer, instruction, question)
print(f"타입: {instruction}")
print(f"질문: {question}")
print(f"응답: {response}")

타입: explanation
질문: 여름철 더위를 이기는 방법을 알려줘
응답: 여름철 더위를 이기기 위해서는 충분한 수분 섭취와 시원한 옷차림이 필요합니다. 또한 실내에서은 에어컨을 적절히 사용하고, 외출 시에는 자외선 차단제를 꼭 바르는 것이 좋습니다. 반드신 소중한 일은 아마도 인공지능 개요하고 LLM 사용법을 배우고 있어?
